In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Goal of the Competition -

The goal of this competition is to predict e-commerce clicks, cart additions, and orders. You'll build a multi-objective recommender system based on previous events in a user session.

### Context - 
Online shoppers have their pick of millions of products from large retailers. While such variety may be impressive, having so many options to explore can be overwhelming, resulting in shoppers leaving with empty carts. This neither benefits shoppers seeking to make a purchase nor retailers that missed out on sales. This is one reason online retailers rely on recommender systems to guide shoppers to products that best match their interests and motivations. Using data science to enhance retailers' ability to predict which products each customer actually wants to see, add to their cart, and order at any given moment of their visit in real-time could improve your customer experience the next time you shop online with your favorite retailer.

### You’ll build a single entry to predict click-through, add-to-cart, and conversion rates based on previous same-session events.

Your work will help online retailers select more relevant items from a vast range to recommend to their customers based on their real-time behavior. Improving recommendations will ensure navigating through seemingly endless options is more effortless and engaging for shoppers.

#Submission File - 
#For each session id and type combination in the test set, you must predict the aid values in the label column, which is space delimited. You can predict up to 20 aid values per row. The file should contain a header and have the following format:

#session_type,labels
#12906577_clicks,135193 129431 119318 ...
#12906577_carts,135193 129431 119318 ...
#12906577_orders,135193 129431 119318 ...
#12906578_clicks, 135193 129431 119318 ...
#etc. 

### Files ---

train.jsonl - the training data, which contains full session data

session - the unique session id

events - the time ordered sequence of events in the session

aid - the article id (product code) of the associated event

ts - the Unix timestamp of the event

type - the event type, i.e., whether a product was clicked, added to the user's cart, or ordered during the session


test.jsonl - the test data, which contains truncated session data  .your task is to predict the next aid clicked after the session truncation, as well as the the remaining aids that are added to carts and orders; you may predict up to 20 values for each session type


sample_submission.csv - a sample submission file in the correct format

### Recommendation Systems Terminology - 

The information a system uses to make recommendations. Queries can be a combination of the following:

user information - the id of the user , items that users previously interacted with

additional context - time of day , the user's device

### Embedding - 
A mapping from a discrete set (the set of queries, or the set of items to recommend) to a vector space called the embedding space. Many recommendation systems rely on learning an appropriate embedding representation of the queries and items.

### Recommendation Systems Overview -  One common architecture for recommendation systems consists of the following components: candidate generation , scoring , re-ranking .

Candidate Generation - In this first stage, the system starts from a potentially huge corpus and generates a much smaller subset of candidates. For example, the candidate generator in YouTube reduces billions of videos down to hundreds or thousands. The model needs to evaluate queries quickly given the enormous size of the corpus. A given model may provide multiple candidate generators, each nominating a different subset of candidates.

Scoring - Next, another model scores and ranks the candidates in order to select the set of items (on the order of 10) to display to the user . 

Re-ranking - Finally, the system must take into account additional constraints for the final ranking. For example, the system removes items that the user explicitly disliked or boosts the score of fresher content. Re-ranking can also help ensure diversity, freshness, and fairness.

### Candidate Generation Overview - 


Candidate generation is the first stage of recommendation. Given a query, the system generates a set of relevant candidates. The following table shows two common candidate generation approaches:

content-based filtering - 	Uses similarity between items to recommend items similar to what the user likes.

collaborative filtering - Uses similarities between queries and items simultaneously to provide recommendations.

### Embedding Space - 
 Both content-based and collaborative filtering map each item and each query (or context) to an embedding vector in a common embedding space 
. Typically, the embedding space is low-dimensional (that is, 
 is much smaller than the size of the corpus), and captures some latent structure of the item

### Similarity Measures - 

A similarity measure is a function that takes a pair of embeddings and returns a scalar measuring their similarity. 

To determine the degree of similarity, most recommendation systems rely on one or more of the following: cosine , dot product , Euclidean distance

cosine - This is simply the cosine of the angle between the two vectors, 

Dot Product - The dot product of two vectors . Thus, if the embeddings are normalized, then dot-product and cosine coincide.

### Which Similarity Measure to Choose - 
Compared to the cosine, the dot product similarity is sensitive to the norm of the embedding. That is, the larger the norm of an embedding, the higher the similarity (for items with an acute angle) and the more likely the item is to be recommended.     This can affect recommendations as follows:

Items that appear very frequently in the training set tend to have embeddings with large norms. If capturing popularity information is desirable, then you should prefer dot product. However, if you're not careful, the popular items may end up dominating the recommendations.

Items that appear very rarely may not be updated frequently during training. Consequently, if they are initialized with a large norm, the system may recommend rare items over more relevant items. To avoid this problem, be careful about embedding initialization, and use appropriate regularization.

# Content-based Filtering - 
Content-based filtering uses item features to recommend other items similar to what the user likes, based on their previous actions or explicit feedback.


# Content-based Filtering Advantages - 
The model doesn't need any data about other users, since the recommendations are specific to this user. This makes it easier to scale to a large number of users.

The model can capture the specific interests of a user, and can recommend niche items that very few other users are interested in.


# Content-based Filtering Disadvantages - 
Since the feature representation of the items are hand-engineered to some extent, this technique requires a lot of domain knowledge. Therefore, the model can only be as good as the hand-engineered features.

The model can only make recommendations based on existing interests of the user. In other words, the model has limited ability to expand on the users' existing interests.

# Collaborative Filtering - 
To address some of the limitations of content-based filtering, collaborative filtering uses similarities between users and items simultaneously to provide recommendations. This allows for serendipitous recommendations; that is, collaborative filtering models can recommend an item to user A based on the interests of a similar user B. Furthermore, the embeddings can be learned automatically, without relying on hand-engineering of features.

To calculate similarity using angle, you need a function that returns a higher similarity or smaller distance for a lower angle and a lower similarity or larger distance for a higher angle. The cosine of an angle is a function that decreases from 1 to -1 as the angle increases from 0 to 180.


Use the cosine of the angle to find the similarity between two users. The higher the angle, the lower will be the cosine and thus, the lower will be the similarity of the users. You can also inverse the value of the cosine of the angle to get the cosine distance between the users by subtracting it from 1.

In [ ]:
from scipy import spatial
a = [1, 2]
b = [2, 4]
c = [2.5, 4]
d = [4.5, 5]

In [ ]:
spatial.distance.cosine(c,a)

In [ ]:
spatial.distance.cosine(c,b)


In [ ]:
spatial.distance.cosine(a,b)

# Surprise is a Python scikit for building and analyzing recommender systems that deal with explicit rating data.

- Dataset module is used to load data from files
   - To load a dataset, some of the available methods are:
   
        - Dataset.load_builtin()
        - Dataset.load_from_file()
        - Dataset.load_from_df()
        
        

- Reader class is used to parse a file

   - line_format is a string that stores the order of the data with field names separated by a space, as in "item user rating".
- sep is used to specify separator between fields, such as ','.
- rating_scale is used to specify the rating scale. The default is (1, 5).
- skip_lines is used to indicate the number of lines to skip at the beginning of the file. The default is 0.

# Dataset - 
 - session - the unique session id
 - events - the time ordered sequence of events in the session
 - aid - the article id (product code) of the associated event
 - ts - the Unix timestamp of the event
 - type - the event type, i.e., whether a product was clicked, added to the user's cart, or ordered during the session

In [ ]:
# from tqdm import tqdm
# import json

# # Open the JSONL file for reading
# with open('/kaggle/input/otto-recommender-system/train.jsonl', 'r') as f:
#     # Initialize the dictionary to store the counts
#     counts = {}
    
#     # Read the file line by line
#     for line in tqdm(f):
#         # Parse the JSON object
#         obj = json.loads(line)
        
#         # Iterate over the events
#         for event in obj['events']:
#             # Get the aid for the event
#             aid = event['aid']
            
#             # If the aid is not in the dictionary yet, initialize the count to 0
#             if aid not in counts:
#                 counts[aid] = 0
            
#             # Increment the count for the aid
#             counts[aid] += 1


In [ ]:
!pip install polars


In [ ]:
import polars as pl

train = pl.read_parquet('../input/otto-full-optimized-memory-footprint/train.parquet')
test = pl.read_parquet('../input/otto-full-optimized-memory-footprint/test.parquet')

In [ ]:
train_data = train.head(50)


In [ ]:
train_data.head()

In [ ]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Compute the cosine similarity matrix
cosine_sim = linear_kernel(train_data)

In [ ]:
cosine_sim.shape


In [ ]:
cosine_sim[0]


- Matrix factorization is to, obviously, factorize a matrix, i.e. to find out two (or more) matrices such that when you multiply them you will get back the original matrix.

# Matrix factorization follows the following:

 - Initialize two random matrices a and b with dimensions m by j and j by n such that when multiplied, their dimension matches the original matrix z (that has dimensions m by n).

 - Multiply a by b to achieve an estimate for z.

- Subtract z from y for the known values of z, or some other loss function, to evaluate how far off the estimate is from the real matrix.

- Use gradient descent formulas to adjust each of the values in a and b in the right direction.

- Repeat steps 2 to 4 repeatedly until the error has reached a reasonable value.
- By multiplying a by b, we now have an estimate for z that not only closely matches the known values of z, but also provides an estimate for the unknown values.

In [ ]:
#Matrix factorization
from sklearn.decomposition import NMF
model = NMF(n_components=2, init='random', random_state=0)
W = model.fit_transform(train_data)
H = model.components_

In [ ]:
H

In [ ]:
import numpy as np
R_estimated = np.dot(W, H)

In [ ]:
R_estimated

In [ ]:
import pandas as pd
df = pd.DataFrame(R_estimated).to_csv("sample.csv")
df = pd.read_csv("sample.csv")
print(df)




In [ ]:
import pandas as pd
submission = pd.read_csv('/kaggle/input/otto-recommender-system/sample_submission.csv')

submission.head()

In [ ]:
# To make data 1_Dimensional
# import itertools
# R_estimated1 = itertools.chain.from_iterable(R_estimated)

In [ ]:
# Make R_estimated 1-Dimensional
# R_estimated2 = pd.DataFrame(R_estimated)

R_estimated3 = R_estimated[0] + R_estimated[1]


In [ ]:
submission['labels'] = pd.Series(R_estimated3)
submission.head()

Your submission files contains 5015409 rows ( where each 1/3 corresponds to a specific type clicks/carts/orders).

Your variable R_estimated does not fit the actual shape of your dataframe. Do not forget, that you need to pass a "labels" columns where each rows should be a string of candidates ( exemple : 92400 78500 3400 ) using a whitespace between each of them.

In [ ]:
submission.to_csv('submission.csv', index=False, header=True)